# What Patients Really Can't Forgive
### Mining 2,000 drug reviews with an LLM to find which side effects actually drive dissatisfaction

This project uses an LLM to extract structured data (side effects, sentiment, whether the drug helped) from unstructured patient drug reviews, then analyzes what drives low ratings and builds a model to predict patient dissatisfaction.

## 1. Setup

In [4]:
!pip install anthropic

import pandas as pd
import json
import html
import time
from anthropic import Anthropic
from google.colab import userdata

client = Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

## 2. Load and clean the data
Public UCI Drug Review Dataset (patient reviews from Drugs.com). We sample 2,000 reviews and fix HTML-encoded characters.

In [5]:
url = "https://raw.githubusercontent.com/hillt5/DATA607_Final_Project/master/drugsComTrain_raw.csv"
full = pd.read_csv(url)

df = full.sample(2000, random_state=42).reset_index(drop=True)
df['review'] = df['review'].apply(html.unescape)

print("shape:", df.shape)
df.head()

shape: (2000, 7)


,uniqueID,drugName,condition,review,rating,date,usefulCount
0,127888,Phentermine,Weight Loss,"""I started taking Phentermine just a little ov...",10,26-Nov-16,24
1,197702,Desvenlafaxine,Depression,"""I have had depression for years due to situat...",10,25-Jul-09,31
2,40759,Leuprolide,Endometriosis,"""I was actually surprised to learn I had stage...",8,21-Dec-11,31
3,208098,Zyclara,Keratosis,"""Have used this for one week but began to have...",7,20-Jan-13,17
4,161657,Diphenhydramine,Allergic Reactions,"""Experienced an allergic reaction during dinne...",10,11-Apr-15,20


## 3. Extract structured data with an LLM
For each review, the LLM returns JSON with side effects, whether the drug helped, and sentiment. The prompt asks it to capture side effects even when phrased in everyday language.

In [6]:
def extract_from_review(review_text):
    prompt = f"""You are carefully reading a patient's drug review to extract structured data.
Read the ENTIRE review closely. Patients often describe side effects in everyday language
without using the words "side effect" — for example "I couldn't sleep" means insomnia.
Capture ALL of these, phrased as short clinical terms.

Extract as JSON:
- "side_effects": a list of EVERY side effect or negative symptom mentioned (empty list if none)
- "helped": true if the drug worked for them, false if not, null if unclear
- "sentiment": "positive", "negative", or "mixed"

Review: {review_text}

Return ONLY the JSON, nothing else."""
    message = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    )
    clean = message.content[0].text.strip()
    if clean.startswith("```"):
        clean = clean.removeprefix("```json").removeprefix("```")
        clean = clean.removesuffix("```")
    return json.loads(clean.strip())

### Run extraction across all 2,000 reviews
Defensive error handling means one bad response never crashes the batch. Results are saved to CSV so this expensive step only runs once.

In [ ]:
results = []

for i in range(len(df)):
    try:
        extracted = extract_from_review(df['review'][i])
        results.append({
            "drug": df['drugName'][i],
            "condition": df['condition'][i],
            "rating": df['rating'][i],
            "side_effects": extracted["side_effects"],
            "helped": extracted["helped"],
            "sentiment": extracted["sentiment"],
        })
    except Exception as e:
        print(f"FAILED at {i} -> {e}")
    if (i + 1) % 100 == 0:
        print(f"processed {i+1}/{len(df)}")
    time.sleep(0.2)

structured_df = pd.DataFrame(results)
structured_df.to_csv("structured_reviews_2000.csv", index=False)
print("Done:", structured_df.shape)

In [9]:
structured_df = pd.read_csv("structured_reviews_2000.csv")
import ast
structured_df["side_effects"] = structured_df["side_effects"].apply(ast.literal_eval)

> **Note:** the cell above re-runs all 2,000 LLM calls.
. To skip it, load the included dataset instead:
> ```python
> structured_df = pd.read_csv("structured_reviews_2000.csv")
> import ast
> structured_df["side_effects"] = structured_df["side_effects"].apply(ast.literal_eval)
> ```
> (The `ast.literal_eval` line rebuilds the side-effect lists, which CSV stores as text.)

## 4. Validate the extraction
Before trusting the LLM's output, check it against an independent signal: do the sentiment labels line up with the actual star ratings?

In [10]:
structured_df.groupby("sentiment")["rating"].mean()

,rating
sentiment,
mixed,6.478528
negative,2.585227
neutral,6.857143
positive,9.136962


Positive reviews average ~9 stars and negative ~2.6 , a ~6.5-point separation. The LLM's sentiment tracks reality, so the extracted features are trustworthy.

## 5. The insight: which side effects drive low ratings?
For each common side effect, compare the average rating of reviews that mention it against the overall average.

In [11]:
overall_avg = structured_df["rating"].mean()
print(f"Overall average rating: {overall_avg:.2f}\n")

top_effects = ["weight gain", "nausea", "fatigue", "insomnia", "acne", "anxiety", "depression"]

for effect in top_effects:
    mask = structured_df["side_effects"].apply(lambda effects: effect in effects)
    avg = structured_df[mask]["rating"].mean()
    n = mask.sum()
    print(f"{effect:15} avg {avg:.2f}  ({n} reviews)  vs overall {overall_avg:.2f}")

Overall average rating: 6.97

weight gain     avg 5.66  (148 reviews)  vs overall 6.97
nausea          avg 6.37  (162 reviews)  vs overall 6.97
fatigue         avg 5.85  (101 reviews)  vs overall 6.97
insomnia        avg 5.60  (106 reviews)  vs overall 6.97
acne            avg 4.40  (62 reviews)  vs overall 6.97
anxiety         avg 5.19  (53 reviews)  vs overall 6.97
depression      avg 3.93  (67 reviews)  vs overall 6.97


**Finding:** side effects affecting mood and self-image ,depression (3.9), acne (4.4), weight gain (5.7), drag ratings far below the 6.97 average, while physical effects like nausea (6.4) barely move it. Patients tolerate feeling briefly sick; they don't forgive a drug that changes their mood or body.

## 6. Predict patient dissatisfaction
Build features from the extracted data and train a logistic-regression model to predict low-rated reviews (rating ≤ 5).

In [12]:
# build numeric features
model_df = pd.DataFrame()
model_df["num_side_effects"] = structured_df["side_effects"].apply(len)
model_df["helped"] = structured_df["helped"].apply(lambda x: 1 if x == True else 0)
sentiment_dummies = pd.get_dummies(structured_df["sentiment"], prefix="sent").astype(int)
model_df = pd.concat([model_df, sentiment_dummies], axis=1)
model_df["low_rating"] = (structured_df["rating"] <= 5).astype(int)

model_df.head()

,num_side_effects,helped,sent_mixed,sent_negative,sent_neutral,sent_positive,low_rating
0,3,1,0,0,0,1,0
1,3,1,0,0,0,1,0
2,2,1,0,0,0,1,0
3,5,0,0,1,0,0,0
4,0,1,0,0,0,1,0


In [13]:
# build numeric features
model_df = pd.DataFrame()
model_df["num_side_effects"] = structured_df["side_effects"].apply(len)
model_df["helped"] = structured_df["helped"].apply(lambda x: 1 if x == True else 0)
sentiment_dummies = pd.get_dummies(structured_df["sentiment"], prefix="sent").astype(int)
model_df = pd.concat([model_df, sentiment_dummies], axis=1)
model_df["low_rating"] = (structured_df["rating"] <= 5).astype(int)

model_df.head()

,num_side_effects,helped,sent_mixed,sent_negative,sent_neutral,sent_positive,low_rating
0,3,1,0,0,0,1,0
1,3,1,0,0,0,1,0
2,2,1,0,0,0,1,0
3,5,0,0,1,0,0,0
4,0,1,0,0,0,1,0


The model predicts dissatisfaction at **0.79 recall / 0.86 precision** , above the ~ 70% naive baseline. Recall is the key metric here because the classes are imbalanced (~30% low-rated), which makes plain accuracy misleading.

## Conclusion

**What this project does:** turns unstructured patient reviews into structured data with an LLM, surfaces which side effects drive dissatisfaction, and predicts low ratings from the extracted features.

**Key results**
- Mood/appearance side effects (depression, acne, weight gain) hurt ratings far more than physical ones, a pattern that held when scaling from 500 to 2,000 reviews.
- The dissatisfaction model reaches 0.79 recall / 0.86 precision, beating the naive baseline.

**Limitations**
- Side-effect terms aren't normalized (e.g. "headache" vs "headaches"), so counts slightly fragment.
- The model's strongest features are LLM-derived sentiment, so some predictive signal is partly circular.
- Next step: add specific side effects as features and test a tree-based model.
